## DestinE Data Streaming

This service offers compressed climate and era5 data and makes it available via a high quality and memory efficient streaming solution. The [SSIM](https://en.wikipedia.org/wiki/Structural_similarity_index_measure) and the mean relative error serve as quality measures.

<div style='white-space: nowrap', align='center'>

<div style='display:inline-block', align='center'>Era5 2 meter dewpoint temperature (01-01-1940 09:00)<br>
<img src="../images/2d9_og_.jpeg" width="450px"><br><img src="../images/2d9_cp_.jpeg" width="450px"><br>Mean SSIM: 0.996<br>Compression rate 1:13<br>Mean relative error 0.1 %</div>

<div style='display:inline-block', align='center'>Era 5 10 metre U wind component (01-01-1940 09:00)<br>
<img src="../images/10u9_og_.jpeg" width="450px"><br><img src="../images/10u9_cp_.jpeg" width="450px"><br>Mean SSIM: 0.995<br>Compression rate 1:27<br>Mean relative error 0.3 %</div>

</div>


## Prerequisites
### DestinE Platform Credentials

You need to have an account on the [Destination Earth Platform](https://auth.destine.eu/realms/desp/account).

#### ⚠️ Warning: Authorized Access Only
The usage of this notebook and data access is reserved only to authorized user groups.

## Access the Data
With a DESP account you can access the stream data proposed in this tutorial.

In [ ]:
%%capture cap
%run ./auth.py

In [ ]:
output = [line for line in cap.stdout.split('\n') if line.strip()]
refresh_token = output[-2]
token = output[-1]
print(token)

jwt.decode(token, options={"verify_signature": False})

# Imports and general definitions
We start by importing necessary packages. Data access now goes through `dte_zarr`, which exchanges the DESP token for a stream, then a dataset-scoped Zarr token, and opens the result as a lazily-loaded, geo-referenced `xarray.DataArray` backed by `dask` — no ffmpeg decoding involved.

Note: The API token must be set here including the user group. This happens in **Authentification**.

In [ ]:
from dte_zarr import open_stream, get_stream_overview, stream_overview_table
from datetime import datetime
import rioxarray  # noqa
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs

FORMAT = '%Y-%m-%dT%H:%M'
%env 

# Stream overview
The code in the cell below calls the DTE streaming-overview endpoint to receive an overview of all available streams.

In [ ]:
overview = get_stream_overview(token)
stream_overview_table(overview, "Climate DT (2nd Generation)")

# Parameters for stream access

Here the parameters are set to access the data from the service.

*variable*: an abbreviated variable for the data</br>
*program_subset*: a name for the program subest for the data</br>
*start_date*: the time and date to start the stream</br>
*end_date*: the time and date to end the stream</br>
</br>

To select a stream, chose parameter values from the table above, or if you have a *code snippet*, use it to replace the code in the cell below:

In [ ]:
program_subset = "Climate DT (2nd Generation)"
variable = "2d"
start_date = "2020-06-01T15:00"
end_date = "2020-06-01T15:00"

start_date = datetime.strptime(start_date, FORMAT)
end_date = datetime.strptime(end_date, FORMAT)

# Loading the stream

`open_stream()` exchanges the token for the requested stream, then for a Zarr-scoped access token, and returns the whole stream as one geo-referenced `xarray.DataArray` (lazy, backed by `dask`) — already indexed by real `time`/`latitude`/`longitude` coordinates instead of raw pixel positions.

`.sel(time=..., method="nearest")` then picks out a single global frame, matching *start_date*. Since `start_date` and `end_date` are the same in this example, that's the only frame we need — nothing else is downloaded.


In [ ]:
d2m = open_stream(token, program_subset, variable)
frame = d2m.sel(time=start_date, method="nearest").compute()

print(frame["time"].values)
print(frame.shape)

# Attaching the CRS

`frame` is already a geo-referenced `xarray.DataArray` with real `latitude`/`longitude` coordinates, so — unlike the old manual `from_bounds()` transform built from `streamer.bbox()` — `rioxarray` can derive the affine transform directly from those coordinates. We only need to point it at the right dims and set the CRS.

In [ ]:
frame = frame.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
frame = frame.rio.write_crs("EPSG:4326")

frame

# Subselecting regions

With rio.clip() from rioxarray it is also possible to subselect polygons in the georeferenced data. With add_feature(cartopy.feature.BORDERS) it is possible to outline countries as well.

In [ ]:
europe = [
    { 'type': 'Polygon',
        'coordinates':
          [[
            [
              40.47389429569694,
              71.51798987593267
            ],
            [
              -9.863756639532625,
              71.51798987593267
            ],
            [
              -9.863756639532625,
              36.1174025295482
            ],
            [
              40.47389429569694,
              36.1174025295482
            ],
            [
              40.47389429569694,
              71.51798987593267
            ]
          ]]
    }
        ]

dat = frame.rio.clip(geometries=europe, drop=True)

fig=plt.figure()

dat.plot()
plt.show()

In [ ]:
germany = [
    {
        'type': 'Polygon',
        'coordinates':     [[
            [
                2.7283245259702937,
                57.55574030864628
            ],
            [
                2.7283245259702937,
                45.19668374239939
            ],
            [
                17.542851520061646,
                45.19668374239939
            ],
            [
                17.542851520061646,
                57.55574030864628
            ],
            [
                2.7283245259702937,
                57.55574030864628
            ]
        ]]
    }
]

dat = frame.rio.clip(geometries=germany, drop=True)

fig=plt.figure()

dat.plot()
plt.show()